In [1]:
import os
os.listdir()

['bin',
 'boot',
 'dev',
 'etc',
 'home',
 'lib',
 'lib32',
 'lib64',
 'libx32',
 'media',
 'mnt',
 'opt',
 'proc',
 'root',
 'run',
 'sbin',
 'srv',
 'sys',
 'tmp',
 'usr',
 'var',
 '.launch',
 'workspace',
 '.first_boot_complete',
 '.dockerenv',
 'venv',
 'NGC-DL-CONTAINER-LICENSE',
 'cuda-keyring_1.1-1_all.deb']

In [2]:
%pip install kagglehub
%pip install tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 KB 1.1 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.0/230.0 KB 5.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys

sys.path.insert(0,"workspace")
from src.data_leader import load_data
import tensorflow as tf

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_ds , test_ds , val_ds = load_data()

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)
base_model.trainable = False

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.03),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomZoom(
        height_factor=0.05,
        width_factor=0.05
    )
])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224,224,3)),
    data_augmentation,
    tf.keras.layers.Lambda(preprocess_input),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(
        1,activation="sigmoid"
    )
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy","recall","precision"]
)
print("=====================> training...")
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)
print("=====================> evaluation...")
result  = model.evaluate(test_ds,return_dict=True)
print("loss : ",result["loss"])
print("accuracy : ",result["accuracy"])
print("accuracy : ",result["precesion"])

Found 5216 files belonging to 2 classes.
Using 4173 files for training.
Found 5216 files belonging to 2 classes.
Using 1043 files for validation.
Found 624 files belonging to 2 classes.
=====================> training...
Epoch 1/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 9s 41ms/step - accuracy: 0.7759 - loss: 0.4590 - precision: 0.8177 - recall: 0.8881 - val_accuracy: 0.7228 - val_loss: 0.5411 - val_precision: 0.6934 - val_recall: 0.9974
Epoch 2/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.9338 - loss: 0.1862 - precision: 0.9412 - recall: 0.9703 - val_accuracy: 0.7356 - val_loss: 0.5941 - val_precision: 0.7027 - val_recall: 1.0000
Epoch 3/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 4s 31ms/step - accuracy: 0.9416 - loss: 0.1579 - precision: 0.9506 - recall: 0.9708 - val_accuracy: 0.7532 - val_loss: 0.5817 - val_precision: 0.7177 - val_recall: 0.9974
Epoch 4/5
131/131 ━━━━━━━━━━━━━━━━━━━━ 4s 30ms/step - accuracy: 0.9534 - loss: 0.1376 - precision: 0.9663 - recall: 0.9697 - val_accuracy: 0.7965 - v